# 10장 Plotly 응용: 인터랙티브 리포트 만들기

『AI 시대의 파이썬 데이터 시각화』 예제 노트북입니다. 셀 첫 줄의 번호(P·C·B)는 책 지면의 코드 번호와 같습니다. 위에서부터 차례로 실행합니다.

## 준비

새 노트북이면 이 장의 코드보다 먼저 아래 준비 셀을 실행합니다(책의 앞 장과 같은 셀).

In [ ]:
# P05 | Plotly 준비: 라이브러리를 불러오고 기본 모양을 정합니다.
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

print('Plotly:', plotly.__version__)
pio.templates.default = 'plotly_white'               # 흰 배경·옅은 격자
px.defaults.line_dash_sequence = ['solid', 'dash']   # 둘째 그룹은 파선
px.defaults.symbol_sequence = ['circle', 'x']        # 둘째 그룹은 X 점

## 10.1 12개월 자료와 graph_objects

In [ ]:
# C01 | 12개월 자료: 날짜 열이 있는 두 지점의 월별 표를 만듭니다.
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

dates = pd.date_range('2026-01-01', periods=12, freq='MS')   # 매월 1일
sales_a = [100, 120, 150, 130, 140, 160, 150, 170, 180, 160, 190, 200]
sales_b = [70, 90, 100, 120, 110, 130, 140, 150, 160, 170, 160, 180]
marketing_a = [10, 12, 15, 13, 14, 16, 15, 17, 18, 16, 19, 20]
marketing_b = [7, 9, 11, 12, 11, 13, 14, 15, 16, 17, 16, 18]
df = pd.DataFrame({'date': list(dates) * 2,
                   'branch': ['A'] * 12 + ['B'] * 12,
                   'sales': sales_a + sales_b,
                   'marketing': marketing_a + marketing_b})
LABELS = {'date': '월', 'branch': '지점', 'sales': '매출 (백만원)',
          'marketing': '광고비 (백만원)'}
branch_colors = {'A': '#0072B2', 'B': '#D55E00'}   # 지점 색(4장과 같음)
print(len(df), '행 |', df['date'].dtype, '| 지점·월 중복:',
      df.duplicated(['branch', 'date']).any())
print(df.groupby('branch')['sales'].sum())

In [ ]:
# C02 | go.Scatter: 트레이스를 직접 만들어 빈 Figure에 더합니다.
a = df[df['branch'] == 'A']
go_one = go.Figure()
go_one.add_trace(go.Scatter(x=a['date'], y=a['sales'],
                            mode='lines+markers', name='A',
                            line_color=branch_colors['A']))
print('트레이스 수:', len(go_one.data), '|', go_one.data[0].type,
      go_one.data[0].mode)
print('점 개수:', len(go_one.data[0].x))

In [ ]:
# C03 | 두 지점 트레이스: 같은 절차를 반복문으로 A·B에 적용합니다.
branch_dashes = {'A': 'solid', 'B': 'dash'}   # 지점 선 모양(흑백 구분)
go_two = go.Figure()
for branch in ['A', 'B']:
    part = df[df['branch'] == branch]
    go_two.add_trace(go.Scatter(
        x=part['date'], y=part['sales'], mode='lines+markers',
        name=branch, line_color=branch_colors[branch],
        line_dash=branch_dashes[branch],
        hovertemplate='%{y}백만원'))         # 툴팁 문구
go_two.update_layout(title='지점별 월 매출',
                     hovermode='x unified',
                     xaxis_title='월', yaxis_title='매출 (백만원)')
go_two.update_xaxes(tickformat='%-m월')      # 날짜 눈금 표기
go_two.update_yaxes(range=[0, 220])
go_two.show()

### 직접 해 보기

**W10-01** `graph_objects`로 B지점 트레이스를 만듭니다.

1. C02를 새 이름(예: `go_b`)으로 복사해 B지점 행으로 바꾸고, 이름과 색을 B지점에 맞춥니다.
2. `go_b.show()`로 그리고 점 개수와 트레이스 이름을 출력으로 확인합니다.
3. B지점 12월 값을 툴팁으로 읽어 `df`의 같은 행과 대조합니다.

**완료 기준**: 트레이스가 하나이고 점이 12개이며, 이름과 색이 B지점이고, 읽은 12월 값을 원자료와 대조했다.

In [ ]:
# W10-01 — 여기에서 직접 해 봅니다.

## 10.2 서브플롯·축·범례

In [ ]:
# C04 | 서브플롯: 한 Figure를 두 패널로 나눠 추이와 연간 합계를 봅니다.
from plotly.subplots import make_subplots

totals = df.groupby('branch', as_index=False)['sales'].sum()
sub = make_subplots(rows=1, cols=2, horizontal_spacing=0.2,
                    subplot_titles=['월별 추이', '연간 합계'])
for branch in ['A', 'B']:
    part = df[df['branch'] == branch]
    total = totals.loc[totals['branch'] == branch, 'sales'].iloc[0]
    sub.add_trace(go.Scatter(x=part['date'], y=part['sales'],
                             mode='lines+markers', name=branch,
                             line_color=branch_colors[branch],
                             line_dash=branch_dashes[branch]),
                  row=1, col=1)
    sub.add_trace(go.Bar(x=[branch], y=[total], name=branch,
                         marker_color=branch_colors[branch]),
                  row=1, col=2)
sub.show()
print('트레이스 수:', len(sub.data))

In [ ]:
# C05 | 범례 묶기: 같은 지점의 선과 막대를 범례 하나로 함께 숨깁니다.
for trace in sub.data:
    trace.legendgroup = trace.name                 # A끼리, B끼리
    trace.showlegend = (trace.type == 'scatter')   # 범례는 선에만
sub.update_layout(legend_title_text='지점')
for t in sub.data:                             # 트레이스마다 한 줄
    print(t.type, t.legendgroup, t.showlegend)

In [ ]:
# C06 | 패널마다 축: 집계 기간이 다른 두 패널에 따로 라벨·범위를 줍니다.
sub.update_xaxes(title_text='월', tickformat='%-m월', row=1, col=1)
sub.update_yaxes(title_text='월 매출 (백만원)', range=[0, 220],
                 row=1, col=1)
sub.update_xaxes(title_text='지점', row=1, col=2)
sub.update_yaxes(title_text='연간 합계 (백만원)', range=[0, 2100],
                 row=1, col=2)
sub.update_layout(title='지점 매출: 월별 추이와 연간 합계')
sub.show()

### 직접 해 보기

**W10-02** 같은 두 패널을 위아래로 다시 배치합니다.

1. `make_subplots(rows=2, cols=1, …)`로 새 Figure를 만들고, 선은 `row=1, col=1`, 막대는 `row=2, col=1`에 더합니다.
2. C05처럼 범례를 지점마다 한 줄로 묶습니다.
3. C06처럼 두 패널의 축 라벨·범위를 정하되, `row`·`col`을 위치에 맞게(아래 패널은 `row=2, col=1`) 바꿉니다.
4. 범례의 B를 눌러 선과 막대가 함께 숨는지 확인합니다.

**완료 기준**: 트레이스가 4개이고 범례가 2줄이며, B를 누르면 선·막대가 함께 숨고, 두 Y축에 라벨·범위가 있으며, 날짜 축과 지점 축을 공유하지 않았다.

In [ ]:
# W10-02 — 여기에서 직접 해 봅니다.

## 10.3 날짜 축과 범위 탐색

In [ ]:
# C07 | 날짜 축 탐색: 범위 슬라이더와 기간 버튼을 붙입니다.
time_fig = px.line(df, x='date', y='sales', color='branch',
                   line_dash='branch', markers=True, labels=LABELS,
                   color_discrete_map=branch_colors,
                   title='지점별 월 매출: 구간을 좁혀 탐색')
time_fig.update_xaxes(
    tickformat='%-m월', title_text=None, rangeslider_visible=True,
    rangeslider_bgcolor='#F2F2F2', rangeselector=dict(buttons=[
        dict(count=3, label='3개월', step='month', stepmode='backward'),
        dict(step='all', label='전체')]))
time_fig.update_yaxes(range=[0, 220])
time_fig.update_layout(hovermode='x unified')
time_fig.show()

In [ ]:
# C07B | 하반기 재집계: 날짜 조건으로 행을 골라 합계를 다시 냅니다.
half = df[df['date'].dt.month >= 7]
half_totals = half.groupby('branch', as_index=False)['sales'].sum()
print('원자료', len(df), '행 → 하반기', len(half), '행')
print(half_totals)
print(totals)

### 직접 해 보기

**W10-03** 4분기를 확대한 화면과 다시 계산한 합계를 구분합니다.

1. C07의 차트를 노트북에서 열고 슬라이더로 10~12월만 보이게 한 뒤, 10월 두 지점 값을 툴팁으로 읽습니다.
2. C07B처럼 날짜 조건으로 10~12월 행을 골라 행 수를 확인하고, 지점별 합계를 다시 계산합니다.
3. 연간 합계 `totals`가 그대로인지 보고 확대만 했을 때와 무엇이 다른지 한 문장으로 씁니다.

**완료 기준**: 읽은 10월 값을 원자료와 대조했고, 고른 행 수를 확인했으며, 4분기 합계를 따로 계산했고, 연간 합계가 그대로이며, 확대와 재집계의 차이를 썼다.

In [ ]:
# W10-03 — 여기에서 직접 해 봅니다.

## 10.4 HTML 리포트 조립

In [ ]:
# C08 | 연간 합계 차트: 리포트의 두 번째 질문에 답합니다.
bar_fig = px.bar(totals, x='branch', y='sales', color='branch',
                 text_auto=',',                   # 천 단위 쉼표
                 pattern_shape='branch',          # B지점 빗금(흑백)
                 pattern_shape_sequence=['', '/'],
                 labels={**LABELS, 'sales': '연간 합계 (백만원)'},
                 color_discrete_map=branch_colors,
                 title='전체 매출 합계: 기간 확대와 무관')
bar_fig.update_traces(textposition='outside')   # 값은 막대 위에
bar_fig.update_yaxes(range=[0, 2100])
bar_fig.show()

In [ ]:
# C09 | 광고비와 매출: 리포트의 세 번째 질문에 답하는 산점도입니다.
scatter_fig = px.scatter(df, x='marketing', y='sales', color='branch',
                         symbol='branch', hover_data={'date': '|%Y-%m'},
                         labels=LABELS, color_discrete_map=branch_colors,
                         title='광고비와 매출: 24개 관측')
scatter_fig.update_traces(marker_size=10, opacity=0.7)
scatter_fig.show()

In [ ]:
# C10 | HTML 리포트: 세 Figure를 한 문서에 담습니다(Plotly.js는 한 번).
from pathlib import Path

parts = [time_fig.to_html(full_html=False, include_plotlyjs=True),
         bar_fig.to_html(full_html=False, include_plotlyjs=False),
         scatter_fig.to_html(full_html=False, include_plotlyjs=False)]
tot = totals.set_index('branch')['sales']
octo = df[df['date'].dt.month == 10].set_index('branch')['sales']
report = f'''<!doctype html>
<html lang="ko"><head><meta charset="utf-8">
<title>지점 매출 리포트</title></head>
<body style="max-width: 900px; margin: auto; font-family: sans-serif">
<h1>지점 매출 리포트</h1>
<p>자료: 교육용 가상 자료(월 합계, 24행, 단위 백만원).
차트의 조작은 그 차트에만 적용됩니다.</p>
<h2>1. 월별 변화</h2>
<p>10월에는 B지점이 높습니다(B {octo["B"]}, A {octo["A"]}백만원).</p>
{parts[0]}
<h2>2. 연간 규모</h2>
<p>합계는 A {tot["A"]:,}, B {tot["B"]:,}백만원입니다.</p>
{parts[1]}
<h2>3. 광고비와 매출</h2>
<p>함께 커지는 관계는 보이지만 원인은 단정하지 않습니다.</p>
{parts[2]}
</body></html>'''
output_dir = Path('ch10_output')
output_dir.mkdir(exist_ok=True)
report_path = output_dir / 'ch10_report.html'
report_path.write_text(report, encoding='utf-8')
print(report_path)

### 직접 해 보기

**W10-04** 리포트에 4분기 문장을 더해 완성합니다.

1. C10의 두 번째 절에 W10-03에서 구한 4분기 합계와 연간 합계의 차이를 설명하는 문장을 더합니다.
2. `ch10_output/W10-04.html`로 저장하고 10.5절 C11의 `report_path`를 이 파일로 바꿔 검사한 뒤 브라우저에서 새로 엽니다.
3. 첫 차트의 범례와 슬라이더를 조작해도 두 번째·세 번째 차트가 바뀌지 않는지 확인합니다.

**완료 기준**: 차트 3개와 자료 설명·단위가 있고, 더한 문장에 기간·대상·수치·단위가 있으며, 파일 검사와 독립 조작을 확인했고, 원인을 단정하지 않았다.

In [ ]:
# W10-04 — 여기에서 직접 해 봅니다.

## 10.5 리포트 검수

In [ ]:
# C11 | 리포트 검수: 차트 수·Plotly.js 횟수·외부 주소를 파일에서 셉니다.
html_text = report_path.read_text(encoding='utf-8')
print('차트 수:', html_text.count('class="plotly-graph-div"'))
print('Plotly.js 포함:', html_text.count('plotly.js v'), '번')
print('외부 주소(src=):', html_text.count(' src='), '곳')
print('크기:', round(report_path.stat().st_size / 1_000_000, 1), 'MB')
print('산점도 관측 수:', sum(len(t.x) for t in scatter_fig.data))

## 10.6 해석과 점검

## 직접 해 보기 풀이

먼저 위의 연습 칸에서 직접 해 본 뒤에 봅니다. 풀이는 한 가지 예이고, 완료 기준을 만족하면 다른 코드도 됩니다.

### W10-01

In [ ]:
# W10-01 | 풀이: C02를 새 이름으로 복사해 B지점 트레이스를 만듭니다.
b_rows = df[df['branch'] == 'B']
go_b = go.Figure()
go_b.add_trace(go.Scatter(x=b_rows['date'], y=b_rows['sales'],
                          mode='lines+markers', name='B',
                          line_color=branch_colors['B']))
go_b.show()
print('트레이스 수:', len(go_b.data), '| 이름:', go_b.data[0].name,
      '| 점 개수:', len(go_b.data[0].x))

In [ ]:
# 3. 툴팁에서 읽은 12월 값을 원자료와 대조
print(df[(df['branch'] == 'B') & (df['date'].dt.month == 12)])
assert len(go_b.data) == 1 and len(go_b.data[0].x) == 12 and go_b.data[0].name == 'B'
# 답(예): 툴팁의 B지점 12월 값 180백만원이 df의 같은 행과 같다.

### W10-02

In [ ]:
# W10-02 | 풀이: 같은 두 패널을 위아래(2행 1열)로 다시 배치합니다.
annual = df.groupby('branch', as_index=False)['sales'].sum()
w2 = make_subplots(rows=2, cols=1, vertical_spacing=0.2,      # 위아래 패널 사이 간격
                   subplot_titles=['월별 추이', '연간 합계'])
for branch in ['A', 'B']:
    part = df[df['branch'] == branch]
    total = annual.loc[annual['branch'] == branch, 'sales'].iloc[0]
    w2.add_trace(go.Scatter(x=part['date'], y=part['sales'],
                            mode='lines+markers', name=branch,
                            line_color=branch_colors[branch],
                            line_dash=branch_dashes[branch]), row=1, col=1)
    w2.add_trace(go.Bar(x=[branch], y=[total], name=branch,
                        marker_color=branch_colors[branch]), row=2, col=1)
for trace in w2.data:                                  # 2. 범례를 지점마다 한 줄로
    trace.legendgroup = trace.name
    trace.showlegend = (trace.type == 'scatter')
w2.update_xaxes(title_text='월', tickformat='%-m월', row=1, col=1)   # 3. 위 패널
w2.update_yaxes(title_text='월 매출 (백만원)', range=[0, 220], row=1, col=1)
w2.update_xaxes(title_text='지점', row=2, col=1)                     #    아래 패널
w2.update_yaxes(title_text='연간 합계 (백만원)', range=[0, 2100], row=2, col=1)
w2.update_layout(legend_title_text='지점', height=600,
                 title='지점 매출: 월별 추이와 연간 합계')
w2.show()

In [ ]:
print('트레이스 수:', len(w2.data), '| 범례 줄:', sum(bool(t.showlegend) for t in w2.data))
assert len(w2.data) == 4 and sum(bool(t.showlegend) for t in w2.data) == 2
assert w2.layout.xaxis.matches is None and w2.layout.xaxis2.matches is None   # 축을 공유하지 않음
# 4. 범례의 B를 누르면 legendgroup이 같은 선과 막대가 함께 숨습니다.

### W10-03

In [ ]:
# W10-03 | 풀이: 4분기를 확대한 화면과 다시 계산한 합계를 구분합니다.
# 1. C07의 차트에서 슬라이더로 10~12월만 보이게 하고, 10월 툴팁을 읽습니다.
print(df[df['date'].dt.month == 10][['branch', 'sales']])

In [ ]:
# 2. 날짜 조건으로 10~12월 행을 골라 다시 계산
q4 = df[df['date'].dt.month >= 10]
q4_totals = q4.groupby('branch', as_index=False)['sales'].sum()
annual = df.groupby('branch', as_index=False)['sales'].sum()
print('원자료', len(df), '행 → 4분기', len(q4), '행')
print(q4_totals)
print(annual)                                        # 3. 연간 합계는 그대로
assert len(q4) == 6 and q4_totals['sales'].tolist() == [550, 510]
# 3.(예) 슬라이더로 확대하면 보이는 구간만 바뀌고 합계는 계산되지 않는다. 4분기 합계 A 550, B 510백만원은
#   행을 골라 다시 계산해야 얻으며, 연간 합계 A 1,850, B 1,580백만원은 그대로다.

### W10-04

In [ ]:
# W10-04 | 풀이: C10의 두 번째 절에 4분기 문장을 더해 리포트를 완성합니다.
from pathlib import Path

parts = [time_fig.to_html(full_html=False, include_plotlyjs=True),
         bar_fig.to_html(full_html=False, include_plotlyjs=False),
         scatter_fig.to_html(full_html=False, include_plotlyjs=False)]
tot = df.groupby('branch')['sales'].sum()
q4 = df[df['date'].dt.month >= 10]
q4t = q4.groupby('branch')['sales'].sum()       # W10-03의 4분기 합계
octo = df[df['date'].dt.month == 10].set_index('branch')['sales']
report = f'''<!doctype html>
<html lang="ko"><head><meta charset="utf-8">
<title>지점 매출 리포트</title></head>
<body style="max-width: 900px; margin: auto; font-family: sans-serif">
<h1>지점 매출 리포트</h1>
<p>자료: 교육용 가상 자료(월 합계, 24행, 단위 백만원).
차트의 조작은 그 차트에만 적용됩니다.</p>
<h2>1. 월별 변화</h2>
<p>10월에는 B지점이 높습니다(B {octo["B"]}, A {octo["A"]}백만원).</p>
{parts[0]}
<h2>2. 연간 규모</h2>
<p>합계는 A {tot["A"]:,}, B {tot["B"]:,}백만원입니다.
4분기(10~12월)만 다시 계산하면 A {q4t["A"]:,}, B {q4t["B"]:,}백만원으로,
두 지점의 차이는 연간 {tot["A"] - tot["B"]:,}백만원이고 4분기만 보면 {q4t["A"] - q4t["B"]:,}백만원입니다.</p>
{parts[1]}
<h2>3. 광고비와 매출</h2>
<p>함께 커지는 관계는 보이지만 원인은 단정하지 않습니다.</p>
{parts[2]}
</body></html>'''
output_dir = Path('ch10_output')
output_dir.mkdir(exist_ok=True)
report_path = output_dir / 'W10-04.html'
report_path.write_text(report, encoding='utf-8')
print(report_path)

In [ ]:
html_text = report_path.read_text(encoding='utf-8')
print('차트 수:', html_text.count('class="plotly-graph-div"'), '| 4분기 문장:', '4분기(10~12월)' in html_text)
print('Plotly.js 포함:', html_text.count('plotly.js v'), '번 | 외부 주소(src=):', html_text.count(' src='), '곳')   # C11 검사
assert html_text.count('class="plotly-graph-div"') == 3 and '550' in html_text and '510' in html_text
assert html_text.count('plotly.js v') == 1 and html_text.count(' src=') == 0
# 2·3. 브라우저에서 ch10_output/W10-04.html을 새로 열어, 첫 차트의 범례와 슬라이더를 조작해도
#   두 번째·세 번째 차트가 바뀌지 않는지 봅니다(차트끼리 연동되지 않음).